# BioMini Phase I Educational Materials

## Notebook 01 — From Python Classes to Biological Objects
### Python class에서 Biological Domain Object로

이 Notebook의 목표는 매우 작은 Python OOP 예제에서 출발하여,
생물학적 대상을 **domain object**로 표현하는 사고방식을 익히는 것입니다.

BioMini의 실제 production package를 처음부터 복사하지 않습니다.
먼저 작은 교육용 구현을 직접 만들고, 마지막에 canonical `biomini` package와 비교합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. Python `class`와 `instance`의 역할을 설명한다.
2. 생물학적 대상을 Python object로 표현한다.
3. attribute와 method를 구분한다.
4. constructor validation이 왜 필요한지 설명한다.
5. `__repr__`, `__len__` 같은 Python data model method를 사용할 수 있다.
6. 단순 data container와 domain object의 차이를 설명한다.
7. 초기 설계의 한계를 발견하고 다음 abstraction이 왜 필요한지 설명한다.

## 1. Starting Point — 가장 작은 Molecule

가장 먼저 이름만 가지는 분자 객체를 만들어 봅니다.

이 단계에서는 일부러 매우 단순하게 시작합니다.

In [ ]:
class Molecule:
    def __init__(self, name):
        self.name = name

    def get_name(self):
        return self.name

In [ ]:
water = Molecule("water")

print(water.get_name())
print(water)

이 코드는 실행되지만 아직 문제가 있습니다.

- 잘못된 이름도 허용됩니다.
- object를 출력해도 의미 있는 정보가 보이지 않습니다.
- type hint가 없습니다.
- validation이 없습니다.

즉, **실행되는 코드와 잘 설계된 domain model은 다릅니다.**

## 2. 첫 번째 개선 — Validation과 Pythonic 표현

이름은 생물학적 객체의 identity 일부이므로 빈 문자열을 허용하지 않겠습니다.
또한 `__repr__()`을 추가해 object 상태를 쉽게 확인할 수 있게 합니다.

In [ ]:
class ValidationError(ValueError):
    """Educational validation error used in this notebook."""


class Molecule:
    def __init__(self, name: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")

        self.name = name.strip()

    def get_name(self) -> str:
        return self.name

    def __repr__(self) -> str:
        return f"Molecule(name='{self.name}')"

In [ ]:
water = Molecule("water")
glucose = Molecule("  glucose  ")

print(water)
print(glucose)
print(glucose.get_name())

### Validation 확인

정상 입력뿐 아니라 잘못된 입력도 확인합니다.

In [ ]:
invalid_names = ["", "   ", None, 123]

for value in invalid_names:
    try:
        Molecule(value)
    except ValidationError as exc:
        print(f"{value!r:>8} -> {type(exc).__name__}: {exc}")

## 3. Protein을 추가해 보기

Protein도 molecule입니다.

따라서 자연스러운 첫 설계는 inheritance를 이용하는 것입니다.

```text
Molecule
   ↑
Protein
```

In [ ]:
class Protein(Molecule):
    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str) or not sequence.strip():
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = "".join(sequence.split()).upper()

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)

    def __repr__(self) -> str:
        return (
            f"Protein(name='{self.name}', "
            f"sequence='{self.sequence}', length={len(self)})"
        )

In [ ]:
protein = Protein(
    "example_protein",
    " ACDE FG "
)

print(protein)
print("name     :", protein.get_name())
print("sequence :", protein.get_sequence())
print("length   :", len(protein))

여기서 이미 몇 가지 중요한 설계 아이디어가 등장했습니다.

- `Protein`은 `Molecule`의 특성을 재사용합니다.
- sequence 입력은 공백 제거와 대문자 변환을 거칩니다.
- `__len__()` 덕분에 `len(protein)`을 사용할 수 있습니다.
- object가 자신의 기본 유효성을 책임집니다.

하지만 아직 **protein sequence가 실제 amino-acid alphabet인지 검증하지 않습니다.**

## 4. AminoAcid — Biological Constant를 Object로 표현하기

20개의 canonical amino acid를 한 글자 code로 표현하고,
각 residue의 평균 질량을 연결해 봅니다.

여기서는 BioMini Phase I에서 사용하는 residue mass table과 같은 값을 사용합니다.

In [ ]:
class AminoAcid:
    # Average residue masses:
    # free amino-acid average mass minus H2O
    mass_dict = {
        "A": 71.0779, "R": 156.1857, "N": 114.1026, "D": 115.0874,
        "C": 103.1429, "Q": 128.1292, "E": 129.1140, "G": 57.0513,
        "H": 137.1393, "I": 113.1576, "L": 113.1576, "K": 128.1723,
        "M": 131.1960, "F": 147.1738, "P": 97.1152, "S": 87.0773,
        "T": 101.1039, "W": 186.2099, "Y": 163.1732, "V": 99.1310,
    }

    acceptable_codes = set(mass_dict)

    def __init__(self, code: str):
        if not isinstance(code, str):
            raise ValidationError("amino-acid code must be a string")

        normalized = code.upper()

        if normalized not in self.acceptable_codes:
            raise ValidationError(f"invalid amino acid: {normalized}")

        self.code = normalized

    def get_mass(self) -> float:
        return self.mass_dict[self.code]

    def __repr__(self) -> str:
        return f"AminoAcid(code='{self.code}')"

In [ ]:
alanine = AminoAcid("a")

print(alanine)
print("mass:", alanine.get_mass())

## 5. Protein Validation 강화

이제 `AminoAcid.acceptable_codes`를 이용해 Protein sequence가 canonical
20 amino-acid alphabet만 포함하도록 검증합니다.

In [ ]:
class Protein(Molecule):
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str) or not sequence.strip():
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = "".join(sequence.split()).upper()
        self._validate_sequence()

        self.amino_acids = [
            AminoAcid(code)
            for code in self.sequence
        ]

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

    def get_sequence(self) -> str:
        return self.sequence

    def get_amino_acids(self):
        return self.amino_acids

    def __len__(self) -> int:
        return len(self.sequence)

    def __repr__(self) -> str:
        return (
            f"Protein(name='{self.name}', "
            f"sequence='{self.sequence}', length={len(self)})"
        )

In [ ]:
protein = Protein("small_protein", "ACDEFG")

print(protein)
print(protein.get_amino_acids())

In [ ]:
try:
    Protein("bad_protein", "ACDX")
except ValidationError as exc:
    print(type(exc).__name__, ":", exc)

## 6. 간단한 Self-Check

아직 `pytest`를 사용하지 않습니다.
정식 regression testing은 Notebook 08에서 다룹니다.

In [ ]:
p = Protein("test", "ACDE")

assert p.get_name() == "test"
assert p.get_sequence() == "ACDE"
assert len(p) == 4
assert len(p.get_amino_acids()) == 4
assert p.get_amino_acids()[0].code == "A"

print("Notebook 01 self-check: PASS")

## 7. 지금 설계의 한계 찾기

이 시점의 구조는:

```text
Molecule
    ↑
Protein

AminoAcid
```

입니다.

이제 DNA와 RNA를 추가한다고 생각해 봅니다.

아마 다음과 같은 코드가 반복됩니다.

```text
DNA
 ├─ sequence
 ├─ normalization
 ├─ get_sequence()
 └─ __len__()

RNA
 ├─ sequence
 ├─ normalization
 ├─ get_sequence()
 └─ __len__()

Protein
 ├─ sequence
 ├─ normalization
 ├─ get_sequence()
 └─ __len__()
```

여기서 다음 질문이 생깁니다.

> DNA, RNA, Protein 모두에게 공통인 `sequence`라는 computational abstraction을
> 별도의 superclass로 만들 수 없는가?

이 질문이 바로 **Notebook 02 — SequenceMolecule Abstraction**의 출발점입니다.

## 8. Canonical BioMini와 연결

지금까지 작성한 코드는 **교육용 축약 구현**입니다.

실제 BioMini Phase I의 canonical implementation은 다음과 같은 방향으로 더 발전되어 있습니다.

```text
DomainEntity
   ↓
Molecule
   ↓
SequenceMolecule
   ↓
Protein
```

production package에서는 stable ID, metadata, provenance,
schema version 등의 책임도 추가됩니다.

Repository root에서 BioMini를 editable install했다면 다음 cell을 실행할 수 있습니다.

```bash
python -m pip install -e .
```

In [ ]:
try:
    from biomini import Molecule as BioMiniMolecule
    from biomini import Protein as BioMiniProtein

    bm_molecule = BioMiniMolecule("water")
    bm_protein = BioMiniProtein("canonical_example", "ACDE")

    print("Canonical Molecule:", bm_molecule)
    print("Canonical Protein :", bm_protein)
    print("Protein length    :", len(bm_protein))

except ImportError:
    print(
        "BioMini package is not installed in this Python environment.\n"
        "Run this notebook from the repository after:\n"
        "    python -m pip install -e ."
    )

## 9. Educational Code vs Canonical Code

| 구분 | 역할 |
|---|---|
| Notebook code | 개념을 단계적으로 직접 구현하고 설계 이유를 이해 |
| `biomini/` source | 실제 프로젝트의 canonical implementation |
| `tests/` | canonical behavior의 regression protection |
| `docs/` | architecture와 design decision의 공식 설명 |

Notebook의 목적은 production source를 복사하는 것이 아니라,
**왜 production architecture가 그렇게 되었는지 재구성하는 것**입니다.

## 10. What We Learned

이번 Notebook의 흐름:

```text
simple class
    ↓
validation
    ↓
inheritance
    ↓
biological constants
    ↓
domain-specific validation
    ↓
maintainable biological object
```

다음 Notebook에서는 DNA, RNA, Protein의 반복되는 sequence logic을
`SequenceMolecule` abstraction으로 정리합니다.

## 11. Exercises

### Exercise 1
`Molecule`에 `get_capitalized_name()` method를 추가해 보십시오.

### Exercise 2
새 `Protein` object를 만들고 sequence 길이를 출력하십시오.

### Exercise 3
잘못된 amino-acid code를 넣었을 때 어떤 exception이 발생하는지 확인하십시오.

### Exercise 4
`AminoAcid`에 `is_hydrophobic()` method를 직접 추가해 보십시오.
나중에 이런 analysis responsibility를 어디에 둘지 다시 논의합니다.

### Exercise 5 — Design Question
DNA class를 지금 직접 만든다면 Protein과 어떤 코드가 중복될지 적어 보십시오.

이 질문의 답이 Notebook 02의 출발점입니다.